In [1]:
import time, whisper, pyttsx3, base64

In [6]:
#whisper init
whisper_model = whisper.load_model('tiny')


# Initialize the pyttsx3 engine
engine = pyttsx3.init()
voices = engine.getProperty('voices')
engine.setProperty('voice', voices[1].id)  # Select English voice (change index as needed)

c:\Users\ColinFrisch\Anaconda3\envs\demo\Lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=de

In [8]:
def speech_to_text_whisper(audio_file, model_size=""):    

    # Transcribe the audio
    start=time.time()
    result = whisper_model.transcribe(audio_file)
    print(time.time()-start, result["text"])
    return result["text"]

In [29]:
def text_to_speech(text,output_path):
    engine.setProperty('rate', 150)
    engine.save_to_file(text, output_path)
    engine.runAndWait()


## Utilisation

In [9]:
#Speech to text
audio_file_path = "Enregistrement.wav"  # Replace with your audio file
input_text=speech_to_text_whisper(audio_file_path,'tiny')

c:\Users\ColinFrisch\Anaconda3\envs\demo\Lib\site-packages\whisper\model.py:124: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  a = scaled_dot_product_attention(


1.3933892250061035  Can you tell me who there's artist is and tell me a bit about his artwork?


In [30]:
# Text to speech
input_text="helo my boy"
text_to_speech(input_text,'output.wav')

## Autres

In [10]:
def resize_image(image, width=None, height=None, keep_ratio=True):

    # Open the image if a path is provided
    if isinstance(image, str):
        with Image.open(image) as img:
            img = img.copy()  # Avoids issues with closed files
    elif isinstance(image, Image.Image):
        img = image
    else:
        raise ValueError("Invalid input: 'image' must be a file path or PIL.Image.Image object.")

    # Ensure at least one of width or height is specified
    if not width and not height:
        raise ValueError("At least one of 'width' or 'height' must be specified.")

    # Maintain aspect ratio if required
    if keep_ratio:
        original_width, original_height = img.size
        if width and not height:  # Calculate height preserving ratio
            height = int((width / original_width) * original_height)
        elif height and not width:  # Calculate width preserving ratio
            width = int((height / original_height) * original_width)
        img.thumbnail((width, height), Image.LANCZOS)
    else:
        # Resize without preserving aspect ratio
        if not width or not height:
            raise ValueError("Both 'width' and 'height' must be specified when keep_ratio=False.")
        img = img.resize((width, height), Image.LANCZOS)

    return img


def encode_image(img):
    img_resized = resize_image(img, width=600)
    buffered = BytesIO()
    img_resized.save(buffered, format="JPEG")
    encoded_string = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return encoded_string


In [ ]:
from PIL import Image
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image

API_KEY = "51799a71-19b4-4c6a-9efe-87a2c971ad7f"
MODEL = "pixtral-12b-2409"

client = OpenAI(
    base_url="https://api.scaleway.ai/b722dfe7-d92b-4f1b-8c60-cf56b6f7ba5f/v1",
    api_key=API_KEY,
)

prompt="This series of images are taken from a video, as a whole what do they tell me about my surroundings ? Be very concise (20 words max)."

base64_imgs = [encode_image(Image.open('final/media/1.jpg')),encode_image(Image.open('final/media/2.jpg')),encode_image(Image.open('final/media/3.jpg'))]

content = [{"type": "text", "text": prompt},{"type": "text", "text": 'Are there cars in the room ?'}] + [
        {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_img}"},
        } for base64_img in base64_imgs
]


message = [
{
    "role": "system",
    "content": """You are a helpful assistant expert in helping blind people in their day-to-day life.
    You are his eyes so everything you see is from his point of view. Your responses have to be quite short.""",
    },
    
{"role": "user",
"content": content,},
    ]


response = client.chat.completions.create(messages=message,model=MODEL,stream = False)

print(response.choices[0].message.content)

You are in an office setting. There are two people present, one is younger. Computers and papers are on the desks. Windows show buildings outside.
